In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras.datasets import mnist

In [ ]:
(x_train, y_train) , (x_test, y_test)= mnist.load_data()

In [ ]:
x_test = x_test.astype("float32")/255.0
x_train = x_train.astype("float32")/255.0

x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)
y_train = to_categorical(y_train, num_classes=10)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 4.5))
for i in range(30):
    plt.subplot(3, 10, i+1)
    plt.imshow(x_train[i].reshape((28, 28)), cmap=plt.cm.binary)
    plt.axis('off')
plt.subplots_adjust(wspace=-0.1, hspace=-0.1)
plt.show()

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range = 0.10,
    width_shift_range=0.1,
    height_shift_range=0.1
)

In [ ]:
x_train3 = x_train[9,].reshape((1,28,28,1))
y_train3 = y_train[9,].reshape((1,10))
plt.figure(figsize=(15, 4.5))

for i in range(30):
    plt.subplot(3, 10, i+1)
    x_train2, y_train2 = next(datagen.flow(x_train3, y_train3))
    plt.imshow(x_train2[0].reshape((28,28)), cmap=plt.cm.binary)
    plt.axis('off')
    if i==9: x_train3 = x_train[11, ].reshape((1, 28, 28, 1))
    if i==19: x_train3 = x_train[18, ].reshape((1, 28, 28, 1))
plt.subplots_adjust(wspace=-0.1, hspace=-0.1)
plt.show()


In [ ]:
nets = 15
model = [0] * nets
for j in range(nets):
    model[j] = Sequential()
    model[j].add(Conv2D(32, kernel_size=3, activation='relu', input_shape = (28, 28, 1)))
    model[j].add(BatchNormalization())

    model[j].add(Conv2D(32, kernel_size=3, activation='relu'))
    model[j].add(BatchNormalization())
    
    model[j].add(Conv2D(32, kernel_size=5, strides=2, padding='same', activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Dropout(0.4))

    model[j].add(Conv2D(64, kernel_size=3, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Conv2D(64, kernel_size=3, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Conv2D(64, kernel_size=5, strides=2, padding='same', activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Dropout(0.4))

    model[j].add(Conv2D(128, kernel_size=4, activation='relu'))
    model[j].add(BatchNormalization())
    model[j].add(Flatten())
    model[j].add(Dropout(0.4))
    model[j].add(Dense(10, activation='softmax'))

    model[j].compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
annealer = LearningRateScheduler(lambda x: 1e-3 * 0.95 ** x)

history = [0] * nets
epochs = 45

for j in range(nets):
    x_train2, x_val2, y_train2, y_val2 = train_test_split(x_train, y_train, test_size=0.1)
    history[j] = model[j].fit(datagen.flow(x_train2, y_train2, batch_size=64),
                    epochs = epochs, steps_per_epoch = x_train2.shape[0]//64,
                    validation_data = (x_val2, y_val2), callbacks=[annealer], verbose=0)
    print("CNN {0:d} Epochs={1:d}, Train accuracy={2:.5f}, Validation accuracy={3:.5f}".format(
        j+1, epochs,max(history[j].history['accuracy']), max(history[j].history['val_accuracy'])
    ))

In [ ]:
results = np.zeros((x_test.shape[0], 10))
for j in range(nets):
    results = results + model[j].predict(x_test)
results = np.argmax(results, axis=1)
results = pd.Series(results, name='Label')
submission = pd.concat([pd.Series(range(1, 28001), name='ImageId'), results], axis=1)
submission.to_csv("MNIST-CNN-ENSAMBLE.csv", index=False)


In [ ]:
import os
if not os.path.exists('cnn_models'):
    os.makedirs('cnn_models')

for i, m in enumerate(model):
    model_name = f"cnn_models/cnn_member_{i}.keras"
    m.save(model_name)
    print(f"Saved {model_name}")